# Justice League Stellar Merger History

## Charlotte Christensen, October 23 2025

This jupyter notebook uses the output Anna Wright's code as modified by Nithun Selva (/home/christenc/Code/python/NithunSelva_startrace/RomZoomSHAnalysisScripts) to find the ID of the last satellite to merger into the main halo branch

In [3]:
import pynbody as pb
import numpy as np
import pandas as pd
import glob
import os
import h5py
import tangos

# For importing modules
import importlib.util
import sys
from pathlib import Path

import tangos.examples.mergers as mergers

In [4]:
#After running .bashrc to set up tangos environment, run the server with:
# tangos serve
# in the terminal


# can supposedly be done from within jupyter notebook with:
#!tangos serve

In [5]:
# Import modules

base_path = '/home/christenc/Code/python/NithunSelva_startrace'

for root, dirs, files in os.walk(base_path):
    if root not in sys.path:
        sys.path.append(root)

## Set up for specific halo

In [6]:
simpath = '/data/REPOSITORY/romulus_zooms'
os.environ['TANGOS_SIMULATION_FOLDER'] = simpath
# os.environ['TANGOS_DB_CONNECTION'] = simpath + 'rom25_dwarf_zooms.db'
tangos.core.init_db(simpath + '/rom25_dwarf_zooms.db')
#tangos.all_simulations()

# Completed 442, 492, 515, 523, 544, 552, 568, 569,  597, 613, 634, 642, 656, 718, 719, 741, 749, 761, 886, 916, 
# 918, 968, 977
simroot = 'r502' # Error on 431; 571, 614, 615, 707
# Hunter's Sample: r468 r488 r489 r502 r555 r556 r563 r615 r618 r716 r753 r850 r852 r1023 
#simroot = 'r1023'
simname = simroot + '.romulus25.3072g1HsbBH'

In [7]:
# Read in the unique halo ids assigned for all star particles
sh_path = '/home/awright/dwarf_stellar_halos/' # Anna's sample. These have disks
sh_path = '/home/hlc73/projects/dwarf_stellar_halos/' # Hunter's sample. These are elliptical
write_path = '/home/christenc/Code/Datafiles/stellar_halos/'

with h5py.File(sh_path+'/'+simroot+'/'+simroot+'_allhalostardata.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] # unique host IDs
    partids = f['particle_IDs'][:] # iords
    pct = f['particle_creation_times'][:] # formation times
    ph = f['particle_hosts'][:] # local host IDs (i.e., host at formation time)
    pp = f['particle_positions'][:] # position at formation time
    tsloc = f['timestep_location'][:] # snapshot where star particle first appears
uIDs = np.unique(hostids)
print(f"Unique host IDs: {uIDs}")

Unique host IDs: ['0136_14' '0136_22' '0186_13' '0192_2' '0192_21' '0192_6' '0223_25'
 '0223_9' '0288_11' '0288_14' '0345_28' '0345_39' '0384_37' '0454_16'
 '0480_-1' '0480_15' '0480_5' '0576_38' '0576_90' '0672_5' '0672_9'
 '0775_2' '0775_24' '1056_11' '1344_11' '1344_8' '1440_25' '1536_15'
 '1632_171' '1632_2' '1632_26' '2208_6' '4032_-1' '4096_1' '4096_2'
 '4096_3' '4096_5']


In [8]:
# Read in tangos database
sim = tangos.get_simulation(simname)

# Determine the main progenitor of the main halo
mh_prog,mh_dbid = sim[-1][1].calculate_for_progenitors('halo_number()','dbid()')
mh_halos = [tangos.get_halo(dbid) for dbid in mh_dbid]

#for mh_halo in mh_halos:
#    print(f"Main Halo Progenitor: {mh_halo}")

In [9]:
# For each of the unique halo ids, 
halo_dict = {}
for uID in uIDs:
    # uID = '0384_10'
    sat_ts = uID.split('_')[0]
    print(f"{simname}/%{sat_ts})")
    sat_hid = int(uID.split('_')[1])
    halo_dict[uID] = uID
    if sat_hid <= 0:
        print('Skipping',uID)
        continue
    db_halo = tangos.get_halo(f"{simname}/%{sat_ts}/halo_{sat_hid}")
    # trace forward to get the merger history
    sat_desc,sat_dbid = db_halo.calculate_for_descendants('halo_number()','dbid()')   
    # sat_desc = sat_desc[::-1]  
    # sat_time = sat_time[::-1]
    sat_halos = [tangos.get_halo(dbid) for dbid in sat_dbid][::-1] # reverse
    # print(f"Satellite Halo: {sat_halos}, Total Descendants: {len(sat_desc)}")               

    # Identify the first most recent time when the descendent of the satellite
    # is not the main halo main progenitor
    for i in range(len(sat_halos)):
        if sat_halos[i] != mh_halos[i]:
            # sat_id = f'{sat_halos[i].timestep.extension[-4:]}_{sat_halos[i].halo_number()}'
            mh_id = f'{sat_halos[i].timestep.extension[-4:]}_{sat_halos[i].halo_number}'
            if mh_id in uIDs:
                if uID != mh_id:
                    print(f"Merger Identified at {sat_halos[i].timestep.extension[-6:]}: Sat Halo {uID} merges into Main Halo Progenitor {mh_id}")
            else:
                print(f"Merger into new halo at {sat_halos[i].timestep.extension[-6:]}: Sat Halo {uID} merges into Main Halo Progenitor {mh_id}, but not in uIDs")
            # if i > 0:
            #     print(f"Merger Identified at {sat_halos[i-1].timestep.extension[-6:]}: Sat Halo {sat_halos[i-1].halo_number} merges into Main Halo Progenitor {mh_halos[i-1].halo_number}")
            halo_dict[uID] = mh_id
            break

    #if i != 0:
    #    print(i, sat_desc[i-1], mh_prog[i-1], sat_time[i-1])
    # print(i, sat_desc[i], mh_prog[i], sat_time[i])

r502.romulus25.3072g1HsbBH/%0136)
Merger Identified at 004096: Sat Halo 0136_14 merges into Main Halo Progenitor 4096_5
r502.romulus25.3072g1HsbBH/%0136)
Merger Identified at 001632: Sat Halo 0136_22 merges into Main Halo Progenitor 1632_2
r502.romulus25.3072g1HsbBH/%0186)
r502.romulus25.3072g1HsbBH/%0192)
r502.romulus25.3072g1HsbBH/%0192)
r502.romulus25.3072g1HsbBH/%0192)
Merger into new halo at 000480: Sat Halo 0192_6 merges into Main Halo Progenitor 0480_4, but not in uIDs
r502.romulus25.3072g1HsbBH/%0223)
r502.romulus25.3072g1HsbBH/%0223)
Merger Identified at 000775: Sat Halo 0223_9 merges into Main Halo Progenitor 0775_2
r502.romulus25.3072g1HsbBH/%0288)
Merger Identified at 001632: Sat Halo 0288_11 merges into Main Halo Progenitor 1632_2
r502.romulus25.3072g1HsbBH/%0288)
Merger into new halo at 001268: Sat Halo 0288_14 merges into Main Halo Progenitor 1268_8, but not in uIDs
r502.romulus25.3072g1HsbBH/%0345)
Merger Identified at 001632: Sat Halo 0345_28 merges into Main Halo Prog

In [10]:
# Print only entries where key != value (i.e., mergers identified)
{k: v for k, v in halo_dict.items() if k != v}

{'0136_14': '4096_5',
 '0136_22': '1632_2',
 '0192_6': '0480_4',
 '0223_9': '0775_2',
 '0288_11': '1632_2',
 '0288_14': '1268_8',
 '0345_28': '1632_2',
 '0345_39': '4096_3',
 '0384_37': '0480_5',
 '0454_16': '0775_2',
 '0576_90': '0775_2',
 '0672_5': '1632_2',
 '1344_8': '1632_2',
 '1632_26': '4096_5'}

In [11]:
if simroot == 'r442':
    halo_dict['0672_5'] = '0672_5' # Manually uncombine MM 442
    halo_dict['0576_-1'] = '1248_2' # Manually combine ghost halo for MM 442
elif simroot == 'r468':
    print("nothing identified to manually adjust for r468") # Without NS
elif simroot == 'r486':
    print("nothing identified to manually adjust for r488") # Without NS
elif simroot == 'r488':
    halo_dict['0096_-2'] = '4096_4'
    halo_dict['3168_9'] = '2976_5'
    halo_dict['1824_9'] = '2112_12'
    halo_dict['0768_2'] = '0969_7'
    halo_dict['0192_-1'] = '0480_2'
    halo_dict['0096_-1'] = '0288_3'
elif simroot == 'r489':
    print("nothing identified to manually adjust for r489") # Without NS
elif simroot == 'r492':
    halo_dict['0273_5'] = '1268_2' # MM 492
    halo_dict['1056_7'] = '1056_7' # MM 492. # Double check that this right
elif simroot == 'r502':
    print("nothing identified to manually adjust for r502") # Without NS
elif simroot == 'r515':
    halo_dict['0192_-2'] = '1440_2' # MM 515
    halo_dict['0288_-1'] = '1440_2' # MM 515
    halo_dict['0480_5'] = '0672_44'
    halo_dict['0480_6'] = '0672_44'
elif simroot == 'r523':
    halo_dict['0480_13'] = '0972_2'
    halo_dict['0576_10'] = '0972_2'
    halo_dict['0672_3'] = '1920_9'
elif simroot == 'r544':
    halo_dict['0288_-4'] = '0288_-1'     # MM 544
    halo_dict['0288_-2'] = '0288_-1'     # MM 544
elif simroot == 'r552':
    print("nothing to manually adjust for r552")
    #halo_dict['0192_-4'] = '1344_6'     # MM 552
elif simroot == 'r555':
    print("nothing identified to manually adjust for r555") # Without NS
elif simroot == 'r556':
    print("nothing identified to manually adjust for r556") # Without NS
elif simroot == 'r563':
    print("nothing identified to manually adjust for r563") # Without NS
elif simroot == 'r568':
    print("nothing to manually adjust for r568")
elif simroot == 'r569':
    print("nothing to manually adjust for r569")
elif simroot == 'r615':
    print("nothing identified to manually adjust for r615") # Without NS
elif simroot == 'r618':
    print("nothing identified to manually adjust for r618") # Without NS
elif simroot == 'r707':
    print("nothing identified to manually adjust for r707") # Without NS
elif simroot == 'r716':
    print("nothing identified to manually adjust for r716") # Without NS
elif simroot == 'r741':
    halo_dict['0768_7'] = '0768_7'
    halo_dict['0096_-3'] = '4096_1'
    halo_dict['0192_-1'] = '0192_8'
elif simroot == 'r753':
    print("nothing identified to manually adjust for r753") # Without NS
elif simroot == 'r761':
    print("nothing identified to manually adjust for r761") # Without NS
elif simroot == 'r850':
    print("nothing identified to manually adjust for r850") # Without NS
elif simroot == 'r852':
    print("nothing identified to manually adjust for r852") # Without NS
elif simroot == 'r886':
    halo_dict['0136_-1'] = '0136_3'
elif simroot == 'r916':
    halo_dict['0136_-1'] = '0186_15'
    halo_dict['0136_-2'] = '0186_15' 
    halo_dict['1152_9'] = '1152_9' # MM 916
elif simroot == 'r918':
#    halo_dict['0096_-2'] = '3168_3'
    halo_dict['0136_6'] = '0136_6'
    halo_dict['0192_11'] = '0192_11'
    halo_dict['0192_14'] = '0192_14'
    halo_dict['0192_17'] = '0192_17'
    halo_dict['0192_35'] = '0192_35'
    halo_dict['0384_10'] = '0384_10'
    halo_dict['0768_33'] = '0768_33'
    halo_dict['1248_44'] = '1248_44'
    halo_dict['1920_85'] = '1920_85'
    halo_dict['2304_12'] = '2304_12'
    halo_dict['0136_-1'] = '3072_3' 
elif simroot == 'r977':
    halo_dict['0096_-1'] = '4096_1' # MM 977
    halo_dict['0384_3'] = '0384_3' # MM 977
    halo_dict['1152_6'] = '1152_6' # MM 977
    halo_dict['1152_9'] = '1152_9' # MM 977


nothing identified to manually adjust for r502


In [12]:
hostids2 = [halo_dict[hid] for hid in hostids]
print(f"Unique host IDs (after adjustment): {np.unique(hostids2)}")

Unique host IDs (after adjustment): ['0186_13' '0192_2' '0192_21' '0223_25' '0480_-1' '0480_15' '0480_4'
 '0480_5' '0576_38' '0672_9' '0775_2' '0775_24' '1056_11' '1268_8'
 '1344_11' '1440_25' '1536_15' '1632_171' '1632_2' '2208_6' '4032_-1'
 '4096_1' '4096_2' '4096_3' '4096_5']


In [13]:
os.makedirs(os.path.join(write_path, simroot), exist_ok=True)
with h5py.File(write_path+'/'+simroot+'/'+simroot+'_allhalostardata_consolidated2.h5','w') as f:
    f.create_dataset('particle_IDs', data=partids) # iords
    f.create_dataset('particle_positions', data=pp) # position at formation time
    f.create_dataset('particle_creation_times', data=pct) # time of formation
    f.create_dataset('timestep_location', data=tsloc) # snapshot where star first appears
    f.create_dataset('particle_hosts', data=ph) # host at that snapshot
    f.create_dataset('host_IDs', data=hostids2) # unique ID of host
print("Data written to :"+write_path+'/'+simroot+'/'+simroot+'_allhalostardata_consolidated2.h5')

Data written to :/home/christenc/Code/Datafiles/stellar_halos//r502/r502_allhalostardata_consolidated2.h5


In [12]:
#!tangos serve

In [13]:
#desc,fid,phid = sim[int(step)][int(halo)].calculate_for_descendants('halo_number()','finder_id()','type()')
#proj,fid,phid = sim[int(unis)][unih].calculate_for_progenitors('halo_number()','finder_id()','type()')